<a href="https://colab.research.google.com/github/sami01979/fer_pattern/blob/main/resnet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available())

2.11.0+cu130 True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/raf
!unzip -q /content/drive/MyDrive/DATASET.zip -d /content/raf
!find /content/raf -maxdepth 3 -type d | sort

/content/raf
/content/raf/DATASET
/content/raf/DATASET/test
/content/raf/DATASET/test/1
/content/raf/DATASET/test/2
/content/raf/DATASET/test/3
/content/raf/DATASET/test/4
/content/raf/DATASET/test/5
/content/raf/DATASET/test/6
/content/raf/DATASET/test/7
/content/raf/DATASET/train
/content/raf/DATASET/train/1
/content/raf/DATASET/train/2
/content/raf/DATASET/train/3
/content/raf/DATASET/train/4
/content/raf/DATASET/train/5
/content/raf/DATASET/train/6
/content/raf/DATASET/train/7


In [ ]:
import os
base = "/content/raf/DATASET"
for split in ["train", "test"]:
    counts = {c: len(os.listdir(f"{base}/{split}/{c}")) for c in sorted(os.listdir(f"{base}/{split}"))}
    print(split, sum(counts.values()))

train 12271
test 3068


In [ ]:
%%writefile train_acc.py
import math, time, argparse, random, os, csv
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torch.utils.flop_counter import FlopCounterMode
from torchvision import datasets, transforms, models
from torchvision.models.resnet import BasicBlock, Bottleneck
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix
from tqdm import tqdm

class ECA(nn.Module):
    def __init__(self, c, gamma=2, b=1):
        super().__init__()
        t = int(abs((math.log2(c) + b) / gamma))
        k = t if t % 2 else t + 1
        self.conv = nn.Conv1d(1, 1, k, padding=k // 2, bias=False)
    def forward(self, x):
        y = F.adaptive_avg_pool2d(x, 1).squeeze(-1).transpose(-1, -2)
        y = self.conv(y).transpose(-1, -2).unsqueeze(-1)
        return x * torch.sigmoid(y)

class SE(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        h = max(c // r, 4)
        self.fc = nn.Sequential(nn.Linear(c, h), nn.ReLU(inplace=True), nn.Linear(h, c))
    def forward(self, x):
        y = self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))
        return x * torch.sigmoid(y)[:, :, None, None]

class CBAM(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        h = max(c // r, 4)
        self.mlp = nn.Sequential(nn.Conv2d(c, h, 1, bias=False), nn.ReLU(inplace=True),
                                 nn.Conv2d(h, c, 1, bias=False))
        self.spatial = nn.Conv2d(2, 1, 7, padding=3, bias=False)
    def forward(self, x):
        a = self.mlp(F.adaptive_avg_pool2d(x, 1))
        m = self.mlp(F.adaptive_max_pool2d(x, 1))
        x = x * torch.sigmoid(a + m)
        s = torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], 1)
        return x * torch.sigmoid(self.spatial(s))

ATTN = {"eca": ECA, "se": SE, "cbam": CBAM}

class AttnBasicBlock(BasicBlock):
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.attn(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

class AttnBottleneck(Bottleneck):
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out = self.attn(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

def load_custom(m, path):
    ckpt = torch.load(path, map_location="cpu")
    sd = ckpt.get("state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
    sd = {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items()}
    own = m.state_dict()
    sd = {k: v for k, v in sd.items() if k in own and own[k].shape == v.shape}
    if len(sd) < 0.5 * len(own):
        raise RuntimeError(f"Only {len(sd)}/{len(own)} tensors matched: wrong weights file for this backbone")
    m.load_state_dict(sd, strict=False)
    print(f"Loaded {len(sd)}/{len(own)} tensors from {path}")

def build_model(backbone, attn, nc, weights_path):
    if backbone == "convnext_tiny":
        if attn != "none" or weights_path:
            raise ValueError("convnext_tiny supports only --attn none and no --weights")
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, nc)
        return m
    if backbone == "resnet18":
        m = models.resnet18(weights=None if weights_path else models.ResNet18_Weights.IMAGENET1K_V1)
        block_cls = AttnBasicBlock
    elif backbone == "resnet50":
        m = models.resnet50(weights=None if weights_path else models.ResNet50_Weights.IMAGENET1K_V2)
        block_cls = AttnBottleneck
    else:
        raise ValueError(backbone)
    if weights_path:
        load_custom(m, weights_path)
    if attn != "none":
        for layer in [m.layer1, m.layer2, m.layer3, m.layer4]:
            for blk in layer:
                ch = blk.bn3.num_features if hasattr(blk, "bn3") else blk.bn2.num_features
                blk.attn = ATTN[attn](ch)
                blk.__class__ = block_cls
    m.fc = nn.Linear(m.fc.in_features, nc)
    return m


MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def get_loaders(root, bs, workers, size, cw_power):
    train_tf = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomAffine(degrees=10, translate=(0.08, 0.08), scale=(0.9, 1.1)),
        transforms.ColorJitter(0.3, 0.3, 0.2),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((size, size)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    full_train = datasets.ImageFolder(f"{root}/train", train_tf)
    full_eval = datasets.ImageFolder(f"{root}/train", eval_tf)
    test_set = datasets.ImageFolder(f"{root}/test", eval_tf)
    targets = np.array(full_train.targets)
    tr_idx, va_idx = train_test_split(np.arange(len(targets)), test_size=0.1,
                                      stratify=targets, random_state=42)
    counts = np.bincount(targets[tr_idx], minlength=len(full_train.classes))
    w = (counts.sum() / (len(counts) * counts)) ** cw_power
    weights = torch.tensor(w, dtype=torch.float)
    kw = dict(num_workers=workers, pin_memory=True)
    return (DataLoader(Subset(full_train, tr_idx), bs, shuffle=True, drop_last=True, **kw),
            DataLoader(Subset(full_eval, va_idx), bs, **kw),
            DataLoader(test_set, bs, **kw), full_train.classes, weights)

@torch.no_grad()
def evaluate(model, loader, device, tta=False):
    model.eval(); ys, ps = [], []
    for x, y in loader:
        x = x.to(device)
        out = model(x).softmax(1)
        if tta:
            out = out + model(torch.flip(x, dims=[3])).softmax(1)
        ps.append(out.argmax(1).cpu()); ys.append(y)
    y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return 100 * (y == p).mean(), 100 * f1_score(y, p, average="macro"), y, p

def fps_of(model, size, device, n=100):
    model = model.to(device).eval()
    x = torch.randn(1, 3, size, size, device=device)
    with torch.no_grad():
        for _ in range(20): model(x)
        if device.type == "cuda": torch.cuda.synchronize()
        t = time.time()
        for _ in range(n): model(x)
        if device.type == "cuda": torch.cuda.synchronize()
    return n / (time.time() - t)

def efficiency(model, size, device):
    params = sum(p.numel() for p in model.parameters()) / 1e6
    cpu_model = model.to("cpu").eval()
    with FlopCounterMode(display=False) as fc:
        cpu_model(torch.randn(1, 3, size, size))
    gmacs = fc.get_total_flops() / 2 / 1e9
    fps_cpu = fps_of(cpu_model, size, torch.device("cpu"), n=10)
    fps_gpu = fps_of(model, size, device) if device.type == "cuda" else float("nan")
    return params, gmacs, fps_cpu, fps_gpu

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--backbone", default="resnet50", choices=["resnet18", "resnet50", "convnext_tiny"])
    ap.add_argument("--attn", default="none", choices=["none", "se", "cbam", "eca"])
    ap.add_argument("--weights", default=None, help="optional face-pretrained weights file (resnet18/resnet50)")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--size", type=int, default=224)
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--warmup", type=int, default=3)
    ap.add_argument("--bs", type=int, default=64)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--wd", type=float, default=0.01)
    ap.add_argument("--cw_power", type=float, default=0.5, help="class-weight strength: 0=off, 1=full")
    ap.add_argument("--select", default="acc", choices=["acc", "f1"], help="validation metric for best epoch")
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--out", default="best_acc.pth")
    ap.add_argument("--log", default="results_acc.csv")
    a = ap.parse_args()

    seed_everything(a.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_dl, val_dl, test_dl, classes, cw = get_loaders(a.data, a.bs, a.workers, a.size, a.cw_power)
    model = build_model(a.backbone, a.attn, len(classes), a.weights).to(device)

    crit = nn.CrossEntropyLoss(weight=cw.to(device), label_smoothing=0.1)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.wd)
    def lr_factor(ep):
        if ep < a.warmup: return (ep + 1) / a.warmup
        return 0.5 * (1 + math.cos(math.pi * (ep - a.warmup) / max(1, a.epochs - a.warmup)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    best = -1.0
    for ep in range(1, a.epochs + 1):
        model.train(); loss_sum = 0.0
        for x, y in tqdm(train_dl, desc=f"Epoch {ep}/{a.epochs}", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            loss_sum += loss.item()
        sched.step()
        v_acc, v_f1, _, _ = evaluate(model, val_dl, device)
        print(f"Epoch {ep}: loss {loss_sum/len(train_dl):.4f} | val acc {v_acc:.2f} | val macro-F1 {v_f1:.2f}")
        score = v_acc if a.select == "acc" else v_f1
        if score > best:
            best = score; torch.save(model.state_dict(), a.out)

    model.load_state_dict(torch.load(a.out, map_location=device))
    t_acc, t_f1, y, p = evaluate(model, test_dl, device)
    t_acc2, t_f12, _, _ = evaluate(model, test_dl, device, tta=True)
    cm = confusion_matrix(y, p)
    per_class = 100 * cm.diagonal() / cm.sum(1)
    tag = f"{a.backbone} | attn {a.attn} | size {a.size} | seed {a.seed}"
    print(f"\n[{tag}] TEST acc {t_acc:.2f} | macro-F1 {t_f1:.2f}")
    print(f"[{tag}] TEST with flip-averaging: acc {t_acc2:.2f} | macro-F1 {t_f12:.2f}")
    print("Per-class acc:", {c: round(float(v), 1) for c, v in zip(classes, per_class)})
    print("Confusion matrix:\n", cm)
    params, gmacs, fps_cpu, fps_gpu = efficiency(model, a.size, device)
    print(f"Params {params:.3f}M | GMACs {gmacs:.3f} | FPS cpu {fps_cpu:.1f} | FPS gpu {fps_gpu:.1f}")

    new = not os.path.exists(a.log)
    with open(a.log, "a", newline="") as f:
        w = csv.writer(f)
        if new: w.writerow(["backbone", "attn", "seed", "size", "cw_power", "weights", "select", "best_val",
                            "test_acc", "test_f1", "test_acc_flip", "test_f1_flip",
                            "params_M", "gmacs", "fps_cpu", "fps_gpu"])
        w.writerow([a.backbone, a.attn, a.seed, a.size, a.cw_power, os.path.basename(a.weights or "imagenet"),
                    a.select, round(best, 2), round(t_acc, 2), round(t_f1, 2), round(t_acc2, 2), round(t_f12, 2),
                    round(params, 3), round(gmacs, 3), round(fps_cpu, 1), round(fps_gpu, 1)])

if __name__ == "__main__":
    main()

Overwriting train_acc.py


In [ ]:
!for a in none eca se cbam; do python train_acc.py --data /content/raf/DATASET --backbone resnet50 --attn $a --seed 0 --out r50_${a}_s0.pth && cp results_acc.csv r50_${a}_s0.pth /content/drive/MyDrive/; done

In [3]:
from google.colab import drive
drive.mount('/content/drive')
!ls -lh /content/drive/MyDrive | grep -E "r50|results_acc"

Mounted at /content/drive
-rw------- 1 root root  91M Oct 31 18:41 r50_eca_s0.pth
-rw------- 1 root root  91M Oct 31 17:52 r50_none_s0.pth
-rw------- 1 root root  312 Oct 31 18:41 results_acc.csv


In [4]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/results_acc.csv")
print(df[["backbone","attn","size","test_acc","test_f1","test_acc_flip","test_f1_flip","params_M","fps_gpu"]])

   backbone  attn  size  test_acc  test_f1  test_acc_flip  test_f1_flip  \
0  resnet50  none   224     85.37    77.24          85.95         78.52   
1  resnet50   eca   224     85.40    78.30          85.63         78.88   

   params_M  fps_gpu  
0    23.522    154.2  
1    23.522    115.5  
